# I7. Imaging and deconvolution

The seventh notebook of the introductory track ({doc}`../index`). It
calibrates A1 small as I6 does, makes a dirty image of the target, cleans it
by hand with Högbom's algorithm, then with GeMClean, rang's imager, and
compares everything with the simulated sky. About a minute.

**Learning goals.** After this notebook you can

- say why the dirty image is the sky convolved with the PSF;
- write Högbom's CLEAN loop and name its parameters: loop gain, threshold,
  number of components;
- explain major and minor cycles and the model, residual and restored
  images;
- run `gemclean` with WSClean-style options, including its stop rule and
  multiscale;
- read an image: noise against the radiometer prediction (I1), flux against
  the truth, and the artefacts of uncalibrated data.

In [ ]:
%matplotlib inline
import json
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS

from rang import core_ms, core_ms_write
from rang import simtruth as st
from rang.notebook import data

os.environ.setdefault("RAYON_NUM_THREADS", "2")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "ms-to-image").is_dir()), None)
WORK = data.default_cache_dir() / "intro-i7"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
COMMANDS = {"rang-core": [sys.executable, "-m", "rang.core_cli"], "gemclean": [sys.executable, "-m", "rang.gemclean"], "python": [sys.executable]}


def sh(line, tail=1):
    print("$", line)
    argv = shlex.split(line)
    p = subprocess.run(COMMANDS.get(argv[0], [argv[0]]) + argv[1:], cwd=WORK, capture_output=True, text=True, encoding="utf-8")
    lines = (p.stdout + p.stderr).rstrip().splitlines()
    print("\n".join(lines[-tail:] if tail else lines))
    assert p.returncode == 0, f"exit {p.returncode}"


sh("python -m rang.simtruth_ms generate small . --workers 2")
text = (ROOT / "examples" / "ms-to-image" / "config.yaml").read_text(encoding="utf-8")
(WORK / "config.yaml").write_text(text.replace("stages: [flag, calibrate, apply, image]", "stages: [flag, calibrate, apply]"), encoding="utf-8")
shutil.copy(ROOT / "examples" / "ms-to-image" / "strategy.yaml", WORK)
sh("rang-core run config.yaml a1-small.ms")  # I6's chain without the image stage
d = st.design("small")
TARGET = 3  # FIELD_ID of SIM-TARGET
COMMON = "-scale 3asec -j 2 -field 3"


def load(name):
    with fits.open(WORK / f"{name}.fits") as f:
        return np.squeeze(f[0].data).astype(float), WCS(f[0].header).celestial


def report(name):
    return json.loads((WORK / f"{name}-report.json").read_text(encoding="utf-8"))

## 1. The dirty image and the PSF

Inverting the sampled visibilities (gridding and an FFT) gives the **dirty
image**. The sampling is incomplete (I3), so the dirty image is the true sky
convolved with the **point spread function** (PSF), the image of a unit
point source made with the same samples and weights. `gemclean` with
`-niter 0` writes both:

In [ ]:
sh(f"gemclean -name dirty -size 1024 1024 {COMMON} -weight briggs 0 -niter 0 a1-small.ms")
dirty, wcs = load("dirty-dirty")
psf, _ = load("dirty-psf")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
axes[0].imshow(dirty, origin="lower", cmap="inferno", vmin=-0.01, vmax=0.05)
axes[0].set_title("dirty image (Jy/beam), 1024 pixels of 3″")
c = np.array(np.unravel_index(np.argmax(psf), psf.shape))
axes[1].imshow(psf[c[0] - 64 : c[0] + 64, c[1] - 64 : c[1] + 64], origin="lower", cmap="RdBu_r", vmin=-0.3, vmax=1)
axes[1].set_title("PSF, central 128 pixels")
plt.show()

A1 small's 16 antennas observe the target in short scans (56 integrations
in 7 scans), so the PSF is elongated and has strong sidelobes. Every source
in the dirty image carries a copy of them.

## 2. Högbom's CLEAN by hand

If the sky is a set of point sources, the PSF's sidelobes can be removed
iteratively (Högbom 1974):

1. find the peak of the residual image (initially the dirty image);
2. add a fraction g (the **loop gain**) of it to the model at that pixel;
3. subtract g × peak × the PSF centred there from the residual;
4. stop at a **threshold** or a maximum number of components.

In [ ]:
def hogbom(dirty, psf, gain=0.1, niter=1000, threshold=0.0, box=896):
    """Högbom CLEAN, peaks searched in the central `box` pixels."""
    res, model = dirty.copy(), np.zeros_like(dirty)
    n = dirty.shape[0]
    cy, cx = np.unravel_index(np.argmax(psf), psf.shape)
    lo, hi = n // 2 - box // 2, n // 2 + box // 2
    for k in range(niter):
        window = np.abs(res[lo:hi, lo:hi])
        y, x = np.unravel_index(np.argmax(window), window.shape)
        y, x = y + lo, x + lo
        peak = res[y, x]
        if abs(peak) <= threshold:
            break
        model[y, x] += gain * peak
        # rows r and columns s of the image overlap the PSF shifted to (y, x)
        r0, r1 = max(0, y - cy), min(n, y - cy + psf.shape[0])
        s0, s1 = max(0, x - cx), min(n, x - cx + psf.shape[1])
        res[r0:r1, s0:s1] -= gain * peak * psf[r0 - y + cy : r1 - y + cy, s0 - x + cx : s1 - x + cx]
    return model, res, k + 1


CUT = slice(None)  # the whole image
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
for ax, niter in zip(axes, (0, 100, 2000)):
    model, res, done = hogbom(dirty, psf, niter=niter) if niter else (None, dirty, 0)
    ax.imshow(res[CUT, CUT], origin="lower", cmap="inferno", vmin=-0.01, vmax=0.05)
    ax.set_title(f"after {done} components: rms {res[CUT, CUT].std() * 1e3:.1f} mJy/beam", fontsize=10)
plt.show()

Where are the components? The truth that wrote A1 knows where every source
is. The 2000-component model, the ten brightest true sources marked:

In [ ]:
sky = st.target_sky(d)
ra, dec = np.array([c["ra_deg"] for c in sky]), np.array([c["dec_deg"] for c in sky])
flux = np.array([c["flux"] for c in sky])
px, py = wcs.world_to_pixel_values(ra, dec)
inside = (np.abs(px - 512) < 448) & (np.abs(py - 512) < 448)
brightest = np.argsort(-flux * inside)[:10]
fig, ax = plt.subplots(figsize=(5.5, 5))
yy, xx = np.nonzero(np.abs(model) > 1e-3)
ax.scatter(xx, yy, s=2000 * np.abs(model[yy, xx]), c="k", label="components above 1 mJy (area ∝ flux)")
ax.plot(px[brightest], py[brightest], "o", mfc="none", mec="C3", ms=12, label="ten brightest true sources")
ax.set(xlim=(0, 1023), ylim=(0, 1023), aspect="equal", xlabel="x (pixel)", ylabel="y (pixel)")
ax.set_title("Högbom model after 2000 components")
ax.legend(loc="lower left", fontsize=8)
plt.show()
print(f"model flux {model.sum():.3f} Jy; true flux of the components in the searched box {flux[inside].sum():.3f} Jy")

This loop works on the gridded image only. Real imagers alternate a
**minor cycle** like the one above, with an approximate PSF on a limited
region, and a **major cycle** that subtracts the model from the
*visibilities* exactly and grids the residual again. The model (clean
components), the **residual** and the **restored image** (the model
convolved with a Gaussian fitted to the PSF's main lobe, plus the residual)
are the products.

## 3. GeMClean

`gemclean` takes WSClean's options. Here: 1024² pixels of 3″, Briggs
robust 0, at most 20,000 components, stopping at 3 times the residual's
robust noise (`-auto-threshold 3`). Each minor cycle ends by GeMClean's
measured stop rule with the floor `-stale-floor 0.01`, the default since
2026-11-03 ({gh}`research/stale-floor.md`):

In [ ]:
sh(f"gemclean -name clean -size 1024 1024 {COMMON} -weight briggs 0 -niter 20000 -auto-threshold 3 a1-small.ms")
run = report("clean")["run"]
print(f"{run['components']} components in {run['major_cycles']} major cycles, stop: {run['stop']}; model flux {run['model_flux']:.3f} Jy")
print(f"residual robust sigma {run['residual_mad_sigma'] * 1e3:.3f} mJy/beam, |residual| peak {run['residual_peak_abs'] * 1e3:.2f} mJy/beam")
pd.DataFrame(
    [{"major cycle": c["index"], "components": c["components"], "first peak (mJy)": round(c["peak_start"] * 1e3, 2), "minor cycle ended by": c["stop"]} for c in run["cycles"]]
)

In [ ]:
images = {k: load(f"clean-{k}")[0] for k in ("dirty", "image", "residual")}
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
for ax, (k, im) in zip(axes, images.items()):
    ax.imshow(im[256:768, 256:768], origin="lower", cmap="inferno", vmin=-0.003, vmax=0.03)
    ax.set_title(f"{k}, central 512 pixels")
plt.show()

## 4. Reading the image against the truth

The true total flux of the target, its brightest source, and what GeMClean
recovered (the restored image's peak within 3 pixels of the true position):

In [ ]:
image, wcs1024 = load("clean-image")
px, py = wcs1024.world_to_pixel_values(ra, dec)
i0 = int(np.argmax(flux))
x0, y0 = int(round(float(px[i0]))), int(round(float(py[i0])))
peak = image[y0 - 3 : y0 + 4, x0 - 3 : x0 + 4].max()
on_image = (px >= 0) & (px < 1024) & (py >= 0) & (py < 1024)
print(f"true flux of all {len(sky)} components {flux.sum():.3f} Jy ({flux[on_image].sum():.3f} Jy inside the image); CLEAN model {run['model_flux']:.3f} Jy")
print(f"brightest source: true {flux[i0]:.3f} Jy at 1284 MHz (spectral index {sky[i0]['alpha']:+.2f}); restored peak {peak:.3f} Jy/beam")

Extended sources (A1 has Gaussians of 20-120″ and an arc) are poorly
described by point components. **Multiscale CLEAN** searches for peaks at
several scales at once:

In [ ]:
sh(f"gemclean -name multiscale -size 1024 1024 {COMMON} -weight briggs 0 -niter 20000 -auto-threshold 3 -multiscale a1-small.ms")
sh(f"gemclean -name natural -size 1024 1024 {COMMON} -weight natural -niter 20000 -auto-threshold 3 a1-small.ms")
rows = []
for name in ("clean", "multiscale", "natural"):
    r = report(name)
    rows.append({"run": name, "weighting": "natural" if name == "natural" else "Briggs 0", "components": r["run"]["components"],
                 "major cycles": r["run"]["major_cycles"], "model flux (Jy)": round(r["run"]["model_flux"], 3),
                 "residual sigma (mJy/beam)": round(r["run"]["residual_mad_sigma"] * 1e3, 3),
                 "beam (″)": f"{r['beam']['bmaj_deg'] * 3600:.1f} × {r['beam']['bmin_deg'] * 3600:.1f}"})
pd.DataFrame(rows)

## 5. Noise: how close to the radiometer equation?

I1 predicted 31 µJy/beam for this target with natural weighting and no
flags. The residual above is more than an order of magnitude higher, and
natural weighting, which should give the lowest thermal noise, gives a
*higher* residual than Briggs 0. The residual is therefore not thermal. One
test: correct DATA with the true Jones terms (I4), including the target-only
gain S that the calibrators cannot measure, keep the flags, and image that
with the same settings.

In [ ]:
shutil.copytree(WORK / "a1-small.ms", WORK / "truth-corrected.ms")
z = np.load(WORK / "truth.npz")
truth = {k.removeprefix("raw_"): z[k] for k in z.files if k.startswith("raw_")}
with core_ms.open_table(str(WORK / "a1-small.ms")) as t:
    a1, a2, field = t.getcol("ANTENNA1"), t.getcol("ANTENNA2"), t.getcol("FIELD_ID")
    raw = t.getcol("DATA")
nbl = len(d.antennas) * (len(d.antennas) - 1) // 2
tindex = np.arange(len(a1)) // nbl
fixed = raw.copy()
sel = field == TARGET
fixed[sel] = st.correct(raw[sel], truth, d, st.Rows(tindex[sel], a1[sel], a2[sel]), z["parallactic_angle"]).astype(np.complex64)
with core_ms_write.open_table(str(WORK / "truth-corrected.ms"), readonly=False) as t:
    t.putcol("CORRECTED_DATA", fixed)
sh(f"gemclean -name truth -size 1024 1024 {COMMON} -weight briggs 0 -niter 20000 -auto-threshold 3 truth-corrected.ms")
for name in ("clean", "truth"):
    r = report(name)["run"]
    print(f"{name:>6}: residual sigma {r['residual_mad_sigma'] * 1e3:.3f} mJy/beam, model flux {r['model_flux']:.3f} Jy")

The difference between the two lines is what calibration left in the
target's data, which includes the gain S (self-calibration's job, I6
section 7). What remains in the truth-corrected image is not calibration
error: unflagged RFI (I5), the weighting and the deconvolution's own errors,
which this notebook does not separate. Exercise 5 starts on that.

## 6. What bad calibration looks like

The same dirty image from DATA, before calibration:

In [ ]:
sh(f"gemclean -name raw -size 1024 1024 {COMMON} -weight briggs 0 -niter 0 -data-column DATA a1-small.ms")
raw_dirty, _ = load("raw-dirty")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, (label, im) in zip(axes, (("DATA (uncalibrated)", raw_dirty), ("CORRECTED_DATA", images["dirty"]))):
    ax.imshow(im[256:768, 256:768], origin="lower", cmap="inferno", vmin=-0.01, vmax=0.05)
    ax.set_title(f"dirty image of {label}")
plt.show()
print(f"peak: {raw_dirty.max():.3f} Jy/beam uncalibrated, {images['dirty'].max():.3f} Jy/beam calibrated")

Without delays, bandpass and gains removed, the visibilities add up out
of phase: the sources smear into the sidelobes and the flux scale is lost.

## Exercises

1. Run `hogbom` with gain 0.5 and 0.02. How many components does each need
   to reach the same residual rms?
2. Run `gemclean` with `-weight uniform` and with `briggs 1`. How do the beam
   and the residual change?
3. Add `-stale-floor 0` (no floor) to the GeMClean call. Do the number of
   major cycles and the residual change? See {gh}`research/stale-floor.md`.
4. Measure the flux of the Gaussian sources in the multiscale and the
   point-component restored images against `target_sky`.
5. Make the truth-corrected image from the noiseless twin (I4) with the
   flags, then from the twin without flags. What residual is left by
   deconvolution alone?

## Where next

- {doc}`i8-polarisation`: Stokes Q, U and V.
- Notebook {doc}`04-imaging` compares weightings and flag sets on real data;
  {doc}`../../concepts/imaging`.
- {doc}`00-ms-to-image`: the image stage of `rang-core run`.
- {gh}`research/clean-theory.md` and {gh}`research/gemclean.md`: how
  GeMClean's minor cycle works and what it was measured against.